# Momentum — Execution Scenario Analysis
Tests how portfolio OOS performance changes depending on when trades are filled.
Signals and trade logic are **never modified** — only the fill price changes.

Hourly data is fetched **once** — swap execution time in cell 2 without re-fetching.

---
### How the baseline works
The engine applies a 1-bar shift: **signal on day T → execute at Close[T]** (midnight UTC = ~1am UK).

`plot_portfolio_oos` computes: `strategy_ret[T] = position[T] * pct_change(Close)[T]`

The scenarios adjust the fill price by modifying `Close` on two specific bar types per trade:

**PRE-ENTRY bar** (last flat bar before trade opens, `position = 0`):
Setting `Close[pre_entry] = exec_price` makes the first holding bar capture `(Close[entry] - exec_price) / exec_price`.
Safe because `position = 0` → this bar contributes zero to returns, so nothing cancels.

**LAST-ACTIVE bar** (last holding bar, next `position = 0`):
Setting `Close[last_active] = exec_price` makes that bar capture `(exec_price - Close[prev]) / Close[prev]`.
Safe because the following bar has `position = 0` → nothing follows to cancel.

| Scenario | Entry fills at | Exit fills at |
|---|---|---|
| `'close'` | Close[T] — midnight UTC (baseline) | Close[T] — midnight UTC |
| `integer` | HH:00 UTC on T+1 (day after signal) | HH:00 UTC on T+1 |
| `'worst_long'` | High of T+1 — worst intraday buy | Low of T+1 — worst intraday sell |

In [ ]:
import os
import sys
import pandas as pd

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows

WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'wf_testing')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos

client = get_binance_client()

---
## 1 — Load OOS results and fetch hourly data (run once)

In [ ]:
# ── load daily OOS pkl files ───────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded daily OOS: {list(coin_dfs.keys())}')

# ── fetch 1h data for every coin (one-time pull) ───────────────────────────────
hourly = {}   # coin → 1h DataFrame

for coin, df in coin_dfs.items():
    symbol = coin + 'USDT'
    start  = str(df.index[0].date())
    end    = str(df.index[-1].date())
    klines = client.get_historical_klines(symbol, '1h', start, end)

    h = pd.DataFrame(klines, columns=[
        'Time','Open','High','Low','Close','Volume',
        'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
    ])
    h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
    for col in ['Open','High','Low','Close']:
        h[col] = h[col].astype(float)
    h = h.set_index('Time')
    hourly[coin] = h
    print(f'  {coin}: {len(h)} hourly bars  ({start} → {end})')

print('\nHourly data ready.')

---
## 2 — Set execution scenario (re-run this cell to switch, no re-fetch needed)

**Options:**
- `SCENARIO = 'close'` — baseline: midnight UTC Close on signal day
- `SCENARIO = 10` — 10am UTC on the day after the signal (any int 0–23)
- `SCENARIO = 'worst_long'` — enter at High[T+1], exit at Low[T+1]

In [ ]:
SCENARIO = 10   # ← 'close'  |  integer hour 0-23  |  'worst_long'


def apply_scenario(coin_dfs, hourly, scenario):
    """
    Adjusts execution price by modifying Close on two specific bar types per trade.

    plot_portfolio_oos computes: strategy_ret[T] = position[T] * pct_change(Close)[T]

    WHY previous approach was wrong:
      Replacing Close on signal days (where position is already non-zero) means
      the exec price appears in both the numerator of bar T and denominator of
      bar T+1. These always cancel: (exec/prev) * (close/exec) = close/prev.

    CORRECT approach:
      PRE-ENTRY bar  (position=0, next position!=0):
        position=0 → bar contributes 0 to returns regardless of Close.
        Close[pre_entry] is the denominator for the NEXT bar's pct_change.
        → first holding bar captures (Close[entry] - exec_price) / exec_price

      LAST-ACTIVE bar (position!=0, next position=0):
        Next bar has position=0 → contributes 0, nothing cancels.
        → that bar captures (exec_price - Close[prev_holding]) / Close[prev_holding]
    """
    if scenario == 'close':
        return coin_dfs

    adjusted = {}

    for coin, df in coin_dfs.items():
        d = df.copy()
        d.index = d.index.normalize()
        pos = d['position']

        # last flat bar before entering a position
        pre_entry   = (pos == 0) & (pos.shift(-1).fillna(0) != 0)
        # last holding bar before going flat
        last_active = (pos != 0) & (pos.shift(-1).fillna(0) == 0)

        exec_close = d['Close'].copy()

        if isinstance(scenario, int):
            h = hourly[coin]
            prices_hh = (
                h[h.index.hour == scenario]['Close']
                .resample('1D').last()
            )
            prices_hh.index = prices_hh.index.tz_localize(None).normalize()
            # shift(-1): value at index T is HH:00 price of day T+1 (execution day)
            next_exec = prices_hh.shift(-1).reindex(d.index).ffill()

            exec_close[pre_entry]   = next_exec[pre_entry]
            exec_close[last_active] = next_exec[last_active]
            print(f'  {coin}: {scenario}h UTC — {int(pre_entry.sum())} entries, {int(last_active.sum())} exits adjusted')

        elif scenario == 'worst_long':
            # High/Low already in the pkl — no API call needed
            # shift(-1): value at T is next day's High / Low
            next_high = d['High'].shift(-1)
            next_low  = d['Low'].shift(-1)
            exec_close[pre_entry]   = next_high[pre_entry]
            exec_close[last_active] = next_low[last_active]
            print(f'  {coin}: worst_long — {int(pre_entry.sum())} entries, {int(last_active.sum())} exits adjusted')

        d['Close'] = exec_close
        adjusted[coin] = d

    return adjusted


coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h_UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'\nScenario applied: {label}')

---
## 3 — Portfolio OOS performance

In [ ]:
SHOW_COINS = ['ETH', 'XRP', 'AVAX', 'SOL', 'BNB']   # or list(coin_dfs.keys())

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},   # benchmark always uses original close
    show       = True,
    save_html  = os.path.join(
        ROOT, 'topics', 'momentum', 'results',
        f'portfolio_{label}.html'
    ),
)

print(f'\n── Scenario: {label} ──')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<20} {v:.4f}')